# Compare Designs across Scenarios

This example demonstrates how to compare designs under conditions that are not known in advance (scenarios) with the `tatc.tradespace` module: the front of each scenario, the share of the scenarios in which each design is efficient, the regret of each design, criteria that judge designs across the scenarios, and a search for designs robust to them.

It continues the example `SearchTradeSpace`, whose design space and study it imports from `trade_space_example.py` (beside the notebooks). The evaluations of every design are taken from the study's store if `SearchTradeSpace` was run before (or are evaluated again, in a few minutes). Its objectives here are the numbers of satellites and of planes and the share of points accurately sampled (whose daily mean is estimated within half the field's standard deviation).

In [1]:
import time
from datetime import timedelta

import pandas as pd

from trade_space_example import ACCURACY, REFERENCE, describe, designs, study
from tatc.tradespace import pareto_front, summarize

# the evaluations of every design, and the designs efficient by accuracy
results = study.run(designs)
accuracy = summarize(results, ["satellites", "planes", "sufficient", "accurate"])
accuracy["description"] = accuracy.apply(describe, axis=1)
accuracy["pareto"] = pareto_front(accuracy, ACCURACY)

## Compare Designs across Scenarios

How accurately samples estimate a daily mean depends on the field: samples hours apart estimate the mean of a field that varies slowly (one that decorrelates in 12 hours) better than that of one that varies within the hour (such as rain rates). Which field the mission will serve is not uncertain as the sample of points or the failures are, which the replicates average over: it is a condition under which to compare designs, a scenario. The `scenarios` of a study give fixed values or uncertain parameters of the design space other values: here, the field's decorrelation time (`decorrelation_time`). The study evaluates every design in each field, with the same samples of points, and records the field as its `scenario`; the field of 3 hours, that of `SearchTradeSpace`, takes its evaluations from the store. Criteria `by` the scenario judge designs in each field, and `robustness` judges them across the fields: the share of the fields in which each design is efficient (its Pareto trace) here, and its regret below:

In [2]:
from tatc.tradespace import Criteria, aggregate, robustness

# fields that decorrelate within an hour (such as rain rates), in 3 hours (as in SearchTradeSpace), and in 12 hours
FIELDS = {
    "1 hour": {"decorrelation_time": timedelta(hours=1)},
    "3 hours": {},
    "12 hours": {"decorrelation_time": timedelta(hours=12)},
}
began = time.perf_counter()
fields = study.replace(scenarios=FIELDS).run(designs)
print(f"{len(fields)} evaluations ({len(FIELDS)} fields) in {time.perf_counter() - began:.0f} s")

# the efficient designs of each field (by accuracy), and the share of the fields in which each design is efficient
BY_FIELD = Criteria(ACCURACY, by="scenario")
by_field = aggregate(fields, BY_FIELD)
by_field["pareto"] = pareto_front(by_field, BY_FIELD)
print("efficient designs:", by_field[by_field.pareto].scenario.value_counts()[list(FIELDS)].to_dict())
field_accuracy = by_field.pivot(index="design", columns="scenario", values="accurate")[list(FIELDS)]
traced = accuracy.set_index("design")[["description", "satellites", "planes"]].join(field_accuracy)
traced["pareto_trace"] = robustness(fields, ACCURACY).set_index("design")["pareto_trace"]
traced[traced["pareto_trace"] > 0.5].sort_values(["satellites", "planes"]).round(2)

16440 evaluations (3 fields) in 155 s
efficient designs: {'1 hour': 14, '3 hours': 13, '12 hours': 6}


,description,satellites,planes,1 hour,3 hours,12 hours,pareto_trace
design,,,,,,,
33b006848799,"800 km, 60 deg FOR, Walker 45 deg: 2 x 1 (spac...",2.0,2.0,26.2,34.2,60.4,0.67
b85daf65c3a4,"800 km, 60 deg FOR, Walker 75 deg: 1 x 3 (spac...",3.0,1.0,8.0,17.0,95.6,0.67
c576e2e2dbd1,"800 km, 60 deg FOR, SSO: 2 at 15:00, 1 at 09:00",3.0,2.0,40.8,60.6,85.8,0.67
3690f1194419,"800 km, 60 deg FOR, SSO: 2 at 15:00, 2 at 09:00",4.0,2.0,69.0,85.4,93.0,0.67
cb6bd361b577,"800 km, 60 deg FOR, Walker 75 deg: 4 x 2 (spac...",8.0,4.0,96.6,96.6,97.8,0.67
5529c21520be,"800 km, 60 deg FOR, Walker 75 deg: 3 x 3 (spac...",9.0,3.0,97.8,97.8,97.8,0.67
de505bc89299,"800 km, 60 deg FOR, Walker 75 deg: 3 x 3 (spac...",9.0,3.0,97.8,97.8,97.8,0.67
679ca275c8fc,"800 km, 60 deg FOR, Walker 75 deg: 3 x 3 (spac...",9.0,3.0,97.8,97.8,97.8,0.67


The 10,960 evaluations of the other two fields take about two minutes here. Which designs are efficient depends on the field. For the field that decorrelates in 12 hours, one plane samples well: three satellites in one plane (inclined 75 degrees) sample 96% of the points accurately, and four of the six efficient designs have one plane (the other two, four satellites in two planes, sample 98%). For the fields that vary within hours, the samples must spread over the day: the same three satellites in one plane sample 17% (3 hours) and 8% (1 hour) accurately, against 61% and 41% for three satellites in two sun-synchronous planes. No design is efficient for every field: of the eight efficient for two fields, seven are efficient for the fields of 1 and 3 hours, and the other, three satellites in one plane, for those of 3 and 12 hours.

A design need not be efficient in any field to be a good choice when the field is not known: its regret in a field is how far it falls behind the best design there. With the numbers of satellites and planes as objectives, regret would count mostly satellites and planes (a design matches the cheapest efficient design by dropping some), so here regret is in the share accurately sampled alone, among the designs of at most a number of satellites (a `Requirement`): the points below the best design of each field with as many satellites at most. The design whose largest regret over the fields is least (minimax regret) is the robust choice:

In [3]:
from tatc.tradespace import Requirement

# the designs of least maximum regret (in points accurately sampled, below the best design of each field), among those
# of at most a number of satellites
robust, robust_design = [], {}
for budget in (2, 3, 4, 6):
    within = Criteria({"accurate": "max"}, requirements=[Requirement("satellites", "<=", budget)])
    regret = robustness(fields, within).set_index("design")
    best = robust_design[budget] = regret["max_regret"].idxmin()
    described = traced.loc[best, ["description", "satellites", "planes"]]
    robust.append({"at most": budget, **described, **regret.loc[best].drop("pareto_trace")})
pd.DataFrame(robust).set_index("at most").round(1)

,description,satellites,planes,max_regret,1 hour regret,3 hours regret,12 hours regret
at most,,,,,,,
2,"800 km, 60 deg FOR, Walker 75 deg: 2 x 1 (spac...",2.0,2.0,11.0,7.6,6.4,11.0
3,"800 km, 60 deg FOR, SSO: 2 at 15:00, 1 at 09:00",3.0,2.0,9.8,9.8,0.4,9.8
4,"800 km, 60 deg FOR, SSO: 2 at 15:00, 2 at 09:00",4.0,2.0,5.0,0.0,0.0,5.0
6,"700 km, 60 deg FOR, Walker 75 deg: 3 x 2 (spac...",6.0,3.0,0.4,0.4,0.2,0.4


The robust choice depends on the budget. With four satellites, two sun-synchronous planes of two (crossing at 09:00 and 15:00) are the best for the fields of 1 and 3 hours and fall 5 points behind the best for the field of 12 hours (93% against 98%, by four satellites in two other planes): a mission that does not yet know its field loses little by choosing them. With three satellites, two sun-synchronous planes of two and one fall 10 points behind three satellites in one plane for the field of 12 hours (86% against 96%) and three in three planes for the field of 1 hour (41% against 51%), and less than a point behind the best for 3 hours. Two satellites fall at least 11 points behind somewhere. Six satellites in three Walker planes (at 700 km) fall at most 0.4 points behind the best of any field, though they are efficient in none: a robust choice need not be the best in any scenario. Regrets of a point or two are within the sampling error of the means of five replicates (standard errors of one to four points, see `SearchTradeSpace`), so the robust designs of four and six satellites are as good as the best of each field.

Criteria `across` the fields (`Across`) judge each design once, by its worst or its mean share over the fields, rather than in each field, which is what a search for designs robust to the field needs (see below). Here every design is evaluated in every field, so the fronts of both are exact:

In [4]:
from tatc.tradespace import Across

# the efficient designs by their worst and by their mean share accurately sampled over the fields (with equal
# weights), and those of the field that decorrelates within an hour
fronts = {}
for rule in ("worst", "mean"):
    criteria = Criteria(ACCURACY, across=Across(rule=rule))
    combined = aggregate(fields, criteria)
    fronts[rule] = set(combined.design[pareto_front(combined, criteria)])
fronts["1 hour"] = set(by_field.design[by_field.pareto & (by_field.scenario == "1 hour")])
print("the field in which each design is least accurate:", field_accuracy.idxmin(axis=1).value_counts().to_dict())
print(f"efficient by the worst: {len(fronts['worst'])}, of which {len(fronts['worst'] & fronts['1 hour'])} in 1 hour")
compared = traced.loc[sorted(fronts["worst"] | fronts["mean"]), ["description", "satellites", "planes", *FIELDS]]
compared["worst"] = compared.index.isin(fronts["worst"])
compared["mean"] = compared.index.isin(fronts["mean"])
compared.sort_values(["satellites", "planes", "mean"]).round(1)

the field in which each design is least accurate: {'1 hour': 1028, '3 hours': 68}
efficient by the worst: 15, of which 12 in 1 hour


,description,satellites,planes,1 hour,3 hours,12 hours,worst,mean
design,,,,,,,,
9b3481aea973,"800 km, 60 deg FOR, SSO: 1 at 12:00",1.0,1.0,3.0,4.2,27.0,True,False
12765c7a29fc,"800 km, 60 deg FOR, SSO: 1 at 15:00",1.0,1.0,3.0,3.6,28.0,True,True
24a159fbc4b7,"800 km, 60 deg FOR, SSO: 2 at 12:00",2.0,1.0,5.6,9.4,70.0,True,False
c9cacc08010d,"800 km, 60 deg FOR, SSO: 2 at 15:00",2.0,1.0,4.8,10.0,73.0,False,True
33b006848799,"800 km, 60 deg FOR, Walker 45 deg: 2 x 1 (spac...",2.0,2.0,26.2,34.2,60.4,True,True
d5d3fa915ca7,"800 km, 60 deg FOR, Walker 45 deg: 1 x 3 (spac...",3.0,1.0,10.2,11.4,67.6,True,False
b85daf65c3a4,"800 km, 60 deg FOR, Walker 75 deg: 1 x 3 (spac...",3.0,1.0,8.0,17.0,95.6,False,True
c576e2e2dbd1,"800 km, 60 deg FOR, SSO: 2 at 15:00, 1 at 09:00",3.0,2.0,40.8,60.6,85.8,True,True
92ff9cd17e00,"800 km, 60 deg FOR, Walker 45 deg: 3 x 1 (spac...",3.0,3.0,50.6,59.8,63.6,True,False


The field that decorrelates within an hour is the hardest for 1,028 of the 1,096 designs (that of 3 hours for the others), so a design's worst share is nearly always its share in that field: 12 of the 15 designs efficient by the worst are efficient in that field, and the worst adds little to it. A worst case suits scenarios that are hardest for different designs; here, the mean and the regret (above) say more. The mean over the fields weighs the field of 12 hours as well, in which one plane samples well: three and four satellites in one plane inclined 75 degrees (96% and 95% in that field) are efficient by the mean, where the worst prefers one plane inclined 45 degrees, about two points better in the field of 1 hour and 28 points worse in that of 12 hours. Eight designs are efficient by both, among them the robust choices of three and four satellites above (in two sun-synchronous planes); the robust choice of six (three Walker planes at 700 km) is efficient by the mean, but not by the worst, by 0.4 points in the field of 1 hour.

## Search for Robust Designs

A search for designs robust to the field evaluates each design that it proposes in every field, and judges it by criteria `across` the fields: here pymoo's, by the mean share accurately sampled over the three fields, with as many proposals as above. Its evaluations are those of the comparison of the fields above, retrieved from the store:

In [5]:
from pymoo.algorithms.moo.nsga2 import NSGA2

from tatc.tradespace import hypervolume, hypervolume_progress, optimize_pymoo

BUDGET = 200  # proposals of the search, as in SearchTradeSpace
# a search for the designs most accurate on average over the fields, against the true front of every design
FIELD_MEAN = Criteria(ACCURACY, across=Across(rule="mean"))
ACCURATE_REFERENCE = {"satellites": REFERENCE["satellites"], "planes": REFERENCE["planes"], "accurate": 0}
field_search = optimize_pymoo(
    study.replace(scenarios=FIELDS), FIELD_MEAN, NSGA2(pop_size=20), ("n_gen", BUDGET // 20), seed=1
)
field_progress = hypervolume_progress(field_search.results, FIELD_MEAN, ACCURATE_REFERENCE)
true_mean = hypervolume(aggregate(fields, FIELD_MEAN), FIELD_MEAN, ACCURATE_REFERENCE)
evaluated = set(field_search.results.design)
print(
    f"{len(evaluated)} designs evaluated (in {len(FIELDS)} fields each): "
    f"{100 * field_progress.iloc[-1] / true_mean:.1f}% of the hypervolume of the true front, "
    f"and {len(evaluated & fronts['mean'])} of its {len(fronts['mean'])} designs"
)

147 designs evaluated (in 3 fields each): 99.3% of the hypervolume of the true front, and 8 of its 12 designs


The search finds 99.3% of the hypervolume of the true front of the mean with 147 designs, and 8 of its 12 designs; each design costs three times as much as in a search of one field, as it is evaluated in each field. Its designs and results refine and confirm its front as for any study, with the same criteria. A search by the worst would search mostly the field of 1 hour here (see above).

## Summary

* The `scenarios` of a study compare designs under conditions rather than averaging over them (here, fields of different decorrelation times, a value of the design space): each design is evaluated in each scenario, with the same replicates, and the evaluations of each scenario are kept apart in the store.
* Criteria `by` the scenario give the front of each, and `robustness` the share of the scenarios in which each design is efficient (its Pareto trace) and its regret in each, whose least maximum is a robust choice (here, six satellites efficient in no field but never half a point behind the best).
* Criteria `across` the scenarios (`Across`) judge each design once, by its worst or its mean over them, as a search for robust designs needs: here the worst is mostly the field that varies fastest, and pymoo's search by the mean finds nearly all of the front of the mean.